# Gating models on their internal representations — a physics-grounded proof of concept

**Project proposal PoC.** This notebook demonstrates the core idea in a clean,
controlled, reproducible way, on two *real governing physical laws*. It runs in a
few minutes on CPU.

## The claim

> A model trained to invert a physical law encodes, in its internal activations,
> whether the current input is *identifiable* (answerable) — and a simple linear
> probe on those activations lets the model **restrain itself**, abstaining when no
> correct answer exists and cutting error on the inputs it keeps.

We show this for **pulse oximetry** (Beer–Lambert → SpO₂) and **cuffless blood
pressure** (Moens–Korteweg → BP), and we are honest about the boundary: *which*
failure modes this catches and which it fundamentally cannot.

## 1. Why these two laws — and what 'unanswerable' means precisely

Both laws are **inverse problems that need a specific input to be solvable**:

| Law | Reads | Identifiable only if | The critical channel |
|-----|-------|----------------------|----------------------|
| Beer–Lambert | SpO₂ = 110 − 25·(R_red / R_ir) | *both* wavelengths present | the **IR** channel |
| Moens–Korteweg | BP = h(PTT, calibration) | subject calibration known | the **calibration** |

Remove or corrupt the critical channel and the problem becomes *non-identifiable*:
many different answers fit what remains. Because we generate the data from the law,
the answerable/unanswerable label is **exact** — not a heuristic. That is the
methodological hook: verifiable ground-truth epistemic labels.

In [ ]:
import sys; from pathlib import Path
for p in ('analysis','.','..'):
    if (Path(p)/'physio_gating_poc.py').exists(): sys.path.insert(0,str(Path(p).resolve())); break
import numpy as np, matplotlib.pyplot as plt
from physio_gating_poc import generate_beer_lambert, generate_moens_korteweg

# Confirm the physics: SpO2 is recoverable from BOTH channels, but not from red alone.
d = generate_beer_lambert(3000, 'clean', seed=0)
R = d['x'][:,0]/d['x'][:,1]
spo2_hat = 110 - 25*R
print(f'clean SpO2 recovery (both channels): corr = {np.corrcoef(spo2_hat, d["y"])[0,1]:.3f}')
print(f'from RED channel alone:              corr = {abs(np.corrcoef(d["x"][:,0], d["y"])[0,1]):.3f}  (confounded by perfusion)')

## 2. The three ways the critical channel can fail

These map to *real clinical failure modes*, and — crucially — to three different
positions relative to the data the model was trained on:

- **missing** (dead sensor): the channel is gone → far off the training manifold.
- **saturated / distorted** (motion, railing): the channel is present but wrong and
  extreme → off the training manifold.
- **mismatched** (IR from a different moment; calibration from a different subject):
  each channel looks *perfectly normal on its own* — only the *combination* is
  inconsistent. This is the plausible-but-wrong case, and it is the hard one.

In [ ]:
kc = 1  # beer: IR is column 1
fig, ax = plt.subplots(1,4, figsize=(13,2.7), sharey=True)
for a,(name,corr) in zip(ax, [('clean','clean'),('missing','missing_ir'),('saturated','saturated_ir'),('mismatched','mismatched_ir')]):
    ir = generate_beer_lambert(400, corr, seed=1)['x'][:,1]
    a.hist(ir, bins=30, color='#2a78d6', alpha=0.8); a.set_title(f'{name}\nIR channel'); a.set_xlabel('R_ir')
ax[0].set_ylabel('count'); plt.tight_layout(); plt.show()
print('missing = spike at 0 (off-manifold); saturated = large (off-manifold); mismatched = looks like clean.')

## 3. Train, freeze, probe, gate

We train a small model to read the target from the clean (answerable) data, freeze
it, then fit a **linear probe on its activations** to flag unanswerable inputs. We
also compute two *label-free* detectors on the same frozen model: **ensemble
disagreement** (spread across independently trained copies) and **OOD distance**
(Mahalanobis in activation space).

The slow cell below runs the whole pipeline for both laws (~2–4 min).

In [ ]:
import argparse
from physio_gating_poc import LAWS, aggregate, plot_law
cfg = argparse.Namespace(seeds=[0,1,2], n_ensemble=6, n_train=4000, n_probe=800,
                         n_test=1000, hidden=64, epochs=300)
out = Path('results/physio_poc'); out.mkdir(parents=True, exist_ok=True)
aggs = {}
for law in ('beer_lambert','moens_korteweg'):
    meta = LAWS[law]
    aggs[law] = aggregate(meta['gen'], meta['corruptions'], cfg)
    plot_law(law, aggs[law], out)
    print(law, 'clean-task MAE =', round(aggs[law]['clean_mae'],2))

### Beer–Lambert result

In [ ]:
from IPython.display import Image
Image(str(out/'poc_beer_lambert.png'))

### Moens–Korteweg result

In [ ]:
Image(str(out/'poc_moens_korteweg.png'))

## 4. What the two panels say

**Left — the model restrains itself.** The risk–coverage curve: as the model
answers fewer inputs (abstaining on the least-confident), the error on what it
*keeps* falls. Gating by the activation probe (blue) tracks the oracle (violet) and
sits far below random abstention (grey). **This is the proof of concept**: a frozen
model, gated on its own internal state, cleanly separates the answerable inputs
from the non-identifiable ones and cuts retained error to near zero at ~50% coverage.

**Right — which failures are catchable.** A clean, honest landscape:

- **Missing channel** → caught by *everything* (it is far off-manifold and has a
  distinctive pattern).
- **Saturated / distorted** → caught by the OOD detector; note the supervised probe
  (trained only on the *missing* failure) can be actively fooled — a warning that a
  probe trained on one failure does not automatically generalize.
- **Mismatched / wrong-subject** → the hard case. It is only *partially* caught,
  and only by the label-free methods, because a plausible mismatch still breaks a
  physical-consistency constraint (the two channels are correlated in real data).
  A corruption that respected *every* observable constraint would be uncatchable —
  the information needed is absent from the input.

## 5. Why this is a good proposal PoC

- **It works and it's clean.** One controlled experiment, real physics, a money
  plot (risk–coverage) that shows the model restraining itself.
- **It has a mechanism, not just a number.** Detectability is governed by the
  *identifiability gap* and by whether the failure moves the input off the data
  manifold (developed formally in `analysis/THEORY.md`, and in the abstract toy
  model `analysis/identifiability_toy_model.ipynb`).
- **It is honest about its boundary.** It says exactly what internal-representation
  gating can do (catch missing/distorted inputs, label-free) and cannot do (catch a
  perfectly in-distribution but wrong input) — which is itself the decision-relevant
  finding for any safety-critical deployment.

### Natural next steps for the project
1. Isolate the *fully-consistent* mismatch to demonstrate the information floor in
   the physiological setting (as the toy model does exactly).
2. Add a third law (Fick, Windkessel) as an out-of-sample test of the protocol.
3. Move from these feature-level simulators to waveform models and, ultimately, one
   real dataset (BIDMC PPG/SpO₂, MIMIC PPG+ABP).
4. Combine the supervised probe with the label-free detectors into a single gate
   and measure coverage at a fixed safe error rate — the deployable quantity.